In [59]:
#importing required libraries
import os
import cv2
import numpy as np
import pandas as pd

from tqdm import tqdm

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [60]:
#path of landmarker
MODEL_PATH = r"C:\Users\hp\Documents\innomatics\DL\Rock_Paper_Scissor\model\hand_landmarker.task"

In [61]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

base_options = python.BaseOptions(
    model_asset_path=MODEL_PATH
)

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1
)

landmarker = vision.HandLandmarker.create_from_options(
    options
)

In [62]:
train_path = r"C:\Users\hp\Documents\innomatics\DL\Rock_Paper_Scissor\dataset\Rock-Paper-Scissors\Rock-Paper-Scissors\train"
test_path = r"C:\Users\hp\Documents\innomatics\DL\Rock_Paper_Scissor\dataset\Rock-Paper-Scissors\Rock-Paper-Scissors\test"
validation_path = r"C:\Users\hp\Documents\innomatics\DL\Rock_Paper_Scissor\dataset\Rock-Paper-Scissors\Rock-Paper-Scissors\validation"

In [63]:
print(os.listdir(train_path))

['paper', 'rock', 'scissors']


In [64]:
#finding length of classes in train data
classes = ['paper', 'rock', 'scissors']


for gesture in classes:
    
    folder_path = os.path.join(
        train_path,
        gesture
    )

    print(
        gesture,
        len(os.listdir(folder_path))
    )

paper 840
rock 840
scissors 840


In [65]:
#finding length of classes in test data
classes = ['paper', 'rock', 'scissors']


for gesture in classes:
    
    folder_path = os.path.join(
        test_path,
        gesture
    )

    print(
        gesture,
        len(os.listdir(folder_path))
    )

paper 124
rock 124
scissors 124


In [66]:
# testing on single image, if detected it gives 1 as output

test_image_path = os.path.join(
    train_path,
    "rock",
    os.listdir(
        os.path.join(train_path, "rock")
    )[0]
)

image = cv2.imread(test_image_path)

image_rgb = cv2.cvtColor(
    image,
    cv2.COLOR_BGR2RGB
)

mp_image = mp.Image(
    image_format=mp.ImageFormat.SRGB,
    data=image_rgb
)

result = landmarker.detect(mp_image)

len(result.hand_landmarks)

1

In [67]:
len(result.hand_landmarks[0])     #hand landmarker has 21 landmarks so it gives 21 as o/p

21

In [68]:
result.hand_landmarks[0][0]      #x,y,z cordinates of single landmark

NormalizedLandmark(x=0.6681621074676514, y=0.7852751016616821, z=-2.737995998813858e-07, visibility=None, presence=None, name=None)

In [69]:
#func to extract landmarks
def extract_landmarks(image_path):

    image = cv2.imread(image_path)

    if image is None:
        return None


    image_rgb = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )


    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=image_rgb
    )


    result = landmarker.detect(mp_image)


    if not result.hand_landmarks:
        return None


    landmarks = result.hand_landmarks[0]


    features = []


    for landmark in landmarks:

        features.extend([
            landmark.x,
            landmark.y,
            landmark.z
        ])


    return features

# 21 landmarks × 3 coordinates = 63 features

In [70]:
# testing landmark extraction on train data

test_image_path = os.path.join(
    train_path,
    "rock",
    os.listdir(
        os.path.join(train_path, "rock")
    )[0]
)


features = extract_landmarks(test_image_path)


print(type(features))

if features:
    print("Number of features:", len(features))

<class 'list'>
Number of features: 63


In [71]:
#extracting features of Training set 
train_data = []


classes = ['paper', 'rock', 'scissors']

for gesture in classes:

    folder_path = os.path.join(
        train_path,
        gesture
    )


    print(f"Processing {gesture}")


    for image_name in tqdm(
        os.listdir(folder_path)
    ):

        image_path = os.path.join(
            folder_path,
            image_name
        )


        landmarks = extract_landmarks(
            image_path
        )


        if landmarks is not None:

            landmarks.append(
                gesture
            )

            train_data.append(
                landmarks
            )

Processing paper


100%|████████████████████████████████████████████████████████████████████████████████| 840/840 [00:29<00:00, 28.20it/s]


Processing rock


100%|████████████████████████████████████████████████████████████████████████████████| 840/840 [00:26<00:00, 31.52it/s]


Processing scissors


100%|████████████████████████████████████████████████████████████████████████████████| 840/840 [00:27<00:00, 30.29it/s]


In [72]:
#extracting features of testing set
test_data = []


classes = ['paper', 'rock', 'scissors']

for gesture in classes:

    folder_path = os.path.join(
        test_path,
        gesture
    )


    print(f"Processing {gesture}")


    for image_name in tqdm(
        os.listdir(folder_path)
    ):

        image_path = os.path.join(
            folder_path,
            image_name
        )


        landmarks = extract_landmarks(
            image_path
        )


        if landmarks is not None:

            landmarks.append(
                gesture
            )

            test_data.append(
                landmarks
            )

Processing paper


100%|████████████████████████████████████████████████████████████████████████████████| 124/124 [00:05<00:00, 23.25it/s]


Processing rock


100%|████████████████████████████████████████████████████████████████████████████████| 124/124 [00:04<00:00, 27.70it/s]


Processing scissors


100%|████████████████████████████████████████████████████████████████████████████████| 124/124 [00:03<00:00, 31.15it/s]


In [73]:
#extracting features of validation set
val_data = []

print("Processing Validation Images")

for image_name in tqdm(os.listdir(validation_path)):

    image_path = os.path.join(validation_path, image_name)

    # Determine label from filename
    if image_name.startswith("paper"):
        gesture = "paper"
    elif image_name.startswith("rock"):
        gesture = "rock"
    elif image_name.startswith("scissors"):
        gesture = "scissors"
    else:
        continue

    landmarks = extract_landmarks(image_path)

    if landmarks is not None:
        landmarks.append(gesture)
        val_data.append(landmarks)

Processing Validation Images


100%|██████████████████████████████████████████████████████████████████████████████████| 33/33 [00:01<00:00, 23.25it/s]


In [74]:
#length of extracted data
print(len(train_data))
print(len(test_data))
print(len(val_data))

2407
369
31


In [75]:
#loss during extracting landmarks. some images landmarks may not be detected and it throws none, which makes data loss

print("Original Train:", len(os.listdir(train_path))*840)      #each folder has 840 rows
print("Extracted Train:", len(train_data))

print("Original Validation:", len(os.listdir(validation_path)))
print("Extracted Validation:", len(val_data))

print("Original Test:", len(os.listdir(test_path))*124)        #each folder has 124 rows
print("Extracted Test:", len(test_data))

Original Train: 2520
Extracted Train: 2407
Original Validation: 33
Extracted Validation: 31
Original Test: 372
Extracted Test: 369


In [76]:
#creating dataframe for train, test and validation 
columns = []

for i in range(63):
    columns.append(
        f"feature_{i}"
    )

columns.append(
    "gesture"
)

train_df = pd.DataFrame(
    train_data,
    columns=columns
)

train_df.head()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_54,feature_55,feature_56,feature_57,feature_58,feature_59,feature_60,feature_61,feature_62,gesture
0,0.559260,0.717897,1.596138e-07,0.430011,0.618903,0.063614,0.349384,0.530109,0.051795,0.277647,...,0.700728,0.328652,-0.154983,0.711742,0.253388,-0.128479,0.711043,0.186671,-0.095309,paper
1,0.561325,0.711195,1.543248e-07,0.427940,0.609307,0.073259,0.345827,0.524454,0.063131,0.275156,...,0.696081,0.323746,-0.152649,0.706822,0.249862,-0.122567,0.707527,0.184072,-0.086675,paper
2,0.557432,0.707801,1.286768e-07,0.421300,0.608627,0.071759,0.337249,0.521762,0.060627,0.265842,...,0.691663,0.315250,-0.160175,0.700977,0.240628,-0.133773,0.699956,0.173105,-0.101476,paper
3,0.551266,0.703698,1.937574e-07,0.409725,0.607470,0.063535,0.324667,0.521406,0.047410,0.250313,...,0.677219,0.306061,-0.164930,0.686671,0.229047,-0.137728,0.687464,0.158137,-0.105616,paper
4,0.541353,0.705843,1.883982e-07,0.401636,0.610321,0.069205,0.315644,0.524243,0.053940,0.243122,...,0.664139,0.306140,-0.194269,0.671247,0.225483,-0.166753,0.671097,0.154222,-0.131608,paper


In [77]:
columns = []

for i in range(63):
    columns.append(
        f"feature_{i}"
    )


columns.append(
    "gesture"
)


test_df = pd.DataFrame(
    test_data,
    columns=columns
)


test_df.head()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_54,feature_55,feature_56,feature_57,feature_58,feature_59,feature_60,feature_61,feature_62,gesture
0,0.622985,0.714573,4.316002e-07,0.462084,0.669217,-0.072020,0.356015,0.583681,-0.102098,0.263318,...,0.637674,0.304626,0.113510,0.621326,0.243875,0.130599,0.605211,0.198659,0.146688,paper
1,0.621660,0.715328,4.598128e-07,0.462532,0.669779,-0.073086,0.356993,0.584939,-0.104373,0.264109,...,0.638886,0.307097,0.108864,0.624422,0.247423,0.124997,0.610177,0.202702,0.140150,paper
2,0.621328,0.714789,4.967511e-07,0.462446,0.668499,-0.071109,0.357692,0.585452,-0.101922,0.264597,...,0.641720,0.308586,0.114968,0.630785,0.249762,0.131510,0.620710,0.204299,0.146643,paper
3,0.622134,0.717794,5.833468e-07,0.461261,0.669052,-0.071762,0.357348,0.586193,-0.103138,0.265200,...,0.645949,0.312740,0.121658,0.637524,0.252301,0.136914,0.631374,0.203416,0.150777,paper
4,0.622402,0.717725,5.870588e-07,0.459793,0.666067,-0.065797,0.355892,0.586217,-0.095609,0.262055,...,0.648161,0.310961,0.119503,0.643073,0.249894,0.134783,0.641045,0.198204,0.149024,paper


In [78]:
columns = []

for i in range(63):
    columns.append(
        f"feature_{i}"
    )


columns.append(
    "gesture"
)


val_df = pd.DataFrame(
    val_data,
    columns=columns
)


val_df.head()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_54,feature_55,feature_56,feature_57,feature_58,feature_59,feature_60,feature_61,feature_62,gesture
0,0.558079,0.656229,-2.885738e-07,0.419787,0.574036,0.078286,0.355398,0.500545,0.115426,0.312924,...,0.658888,0.298920,0.092113,0.666182,0.244999,0.115936,0.675734,0.195526,0.140042,paper
1,0.530397,0.722853,2.002491e-07,0.390859,0.679371,-0.006473,0.314215,0.597311,-0.014379,0.270485,...,0.638095,0.346491,-0.025239,0.644773,0.280805,-0.021243,0.648287,0.229180,-0.015957,paper
2,0.479713,0.897167,-1.332062e-07,0.295331,0.783109,0.026269,0.230330,0.629596,0.032173,0.232895,...,0.659772,0.398643,0.012431,0.663022,0.305135,0.035932,0.661400,0.239201,0.060299,paper
3,0.540480,0.825714,1.521135e-07,0.387636,0.759732,-0.015083,0.307731,0.654308,-0.026952,0.268853,...,0.696092,0.407485,-0.046967,0.708112,0.331360,-0.050238,0.715543,0.270226,-0.051081,paper
4,0.489782,0.727253,5.048663e-07,0.384641,0.680354,-0.017726,0.305598,0.619348,-0.047301,0.233237,...,0.646648,0.452393,-0.101119,0.680978,0.407507,-0.106339,0.709320,0.371129,-0.109470,paper


In [79]:
#checking shape
print(train_df.shape)
print(test_df.shape)
print(val_df.shape)

(2407, 64)
(369, 64)
(31, 64)


In [80]:
CSV_PATH = r"C:\Users\hp\Documents\innomatics\DL\Rock_Paper_Scissor\dataset\train.csv"

train_df.to_csv(
    CSV_PATH,
    index=False
)

In [81]:
CSV_PATH = r"C:\Users\hp\Documents\innomatics\DL\Rock_Paper_Scissor\dataset\test.csv"

test_df.to_csv(
    CSV_PATH,
    index=False
)

In [82]:
CSV_PATH = r"C:\Users\hp\Documents\innomatics\DL\Rock_Paper_Scissor\dataset\validation.csv"

val_df.to_csv(
    CSV_PATH,
    index=False
)